# Week 12 Day 2: RAG + PDF Ingestion + Memory Loops + Conversational Agents
## Complete Practical Notebook — All Code Tested with Expected Outputs

| Part | What You Build | Time |
|------|---------------|------|
| 1 | RAG Pipeline + Hallucination Test | 12 min |
| 2 | PDF Document Ingestion (5 Patient PDFs) | 15 min |
| 3 | Memory Loop + Decay | 12 min |
| 4 | Conversational Agent (PDF + Memory — Capstone) | 15 min |
| 5 | Group Exercise Template | 25 min |
| 6 | End-to-End Integration Test | 5 min |


---\n## Setup

In [ ]:
# !pip install langchain-openai langchain faiss-cpu python-dotenv tiktoken langchain-community pypdf

import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.schema import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()

# Knowledge base — 20 customer support docs
knowledge_base = [
    "Refund policy: Full refund within 30 days. Partial refund (50%) within 60 days.",
    "Shipping: Standard 3-5 days. Express 1-2 days for $9.99.",
    "Premium members get free express shipping on orders over $50.",
    "Password reset: Settings > Security > Reset Password.",
    "Account deletion: Contact support. Data removed within 30 days (GDPR).",
    "Payment: Visa, MasterCard, Amex, PayPal, Apple Pay, Google Pay.",
    "Order tracking: Tracking link emailed after shipping. 24 hours to activate.",
    "International shipping: 45 countries. 7-14 days. Import duties customer responsibility.",
    "Gift returns: Recipients get store credit. Original purchaser gets refund.",
    "Damaged items: Report within 48 hours with photos. Free replacement in 2 days.",
    "Subscriptions: Basic $9.99/mo, Pro $19.99/mo, Enterprise custom.",
    "Cancellation: Cancel anytime, no fee. Refund for unused portion.",
    "Price matching: Match competitor prices within 14 days with proof.",
    "Bulk orders: 50+ units = 15% discount. Contact sales@company.com.",
    "Warranty: Electronics 2-year warranty. Extended warranty $29.99/year.",
    "Store hours: Online 24/7. Support Mon-Fri 8am-8pm EST.",
    "Loyalty: 1 point per dollar. 100 points = $5 discount. Expire after 12 months.",
    "Size exchanges: Free within 30 days. Unworn with tags.",
    "Two-factor auth: Settings > Security > 2FA.",
    "Data privacy: GDPR/CCPA compliant. Full data export available anytime.",
]

# Pre-build FAISS store (reused across parts)
store = FAISS.from_texts(knowledge_base, embeddings)

print("=" * 55)
print("  Day 2 — Environment Ready")
print("=" * 55)
print(f"  LLM         : gpt-4o-mini")
print(f"  Embeddings  : text-embedding-3-small")
print(f"  FAISS store : {store.index.ntotal} docs indexed")
print(f"  API Key     : {'Loaded ✓' if os.getenv('OPENAI_API_KEY') else 'NOT FOUND ✗'}")
print("=" * 55)


---
# Part 1 — RAG Pipeline
### Build → Test → Compare RAG vs Bare LLM

```
User Question → FAISS search (top-3 chunks) → Inject into prompt → LLM answers with source citations
```


In [ ]:
from langchain.chains import RetrievalQA

retriever = store.as_retriever(search_kwargs={"k": 3})
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

# ── Test with 5 questions ──────────────────────────────────
questions = [
    "I bought something 45 days ago. Can I still get a refund?",
    "How do I set up two-factor authentication?",
    "My laptop arrived with a cracked screen. What should I do?",
    "How many loyalty points do I need for a $5 discount?",
    "Can you match a price from Amazon?",
]

print("=" * 60)
print("  RAG PIPELINE — Grounded Answers with Sources")
print("=" * 60)

for q in questions:
    result = rag_chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result'][:150]}...")
    print(f"   Sources: {[d.page_content[:40]+'...' for d in result['source_documents']]}")


**Expected Output:** Each answer cites specific policy documents. The refund question should mention "50% partial refund" for 30-60 days.


In [ ]:
# ── HALLUCINATION TEST: RAG vs Bare LLM ───────────────────
test_q = "What is your refund policy for items returned after 30 days?"

rag_result = rag_chain.invoke({"query": test_q})
bare_result = llm.invoke([HumanMessage(content=test_q)]).content

print("=" * 60)
print("  HALLUCINATION TEST")
print("=" * 60)
print(f"\nQ: {test_q}\n")

print("RAG (grounded — has sources):")
print(f"  {rag_result['result'][:200]}")
print(f"  Source: {rag_result['source_documents'][0].page_content[:60]}...")

print("\nBare LLM (no retrieval — may hallucinate):")
print(f"  {bare_result[:200]}")

has_50 = "50" in rag_result['result']
print(f"\nRAG correctly cites 50% partial refund: {'✓ YES' if has_50 else 'check manually'}")
print("Bare LLM may invent a completely different policy.")


**Expected:** RAG says "50% partial refund within 60 days" (from knowledge base). Bare LLM may hallucinate "full refund" or "no refund" — confident but wrong.


---
# Part 2 — PDF Document Ingestion
### Load Real Patient PDFs → Chunk → Embed → Query with Citations

This is the **crowd-pleaser**: upload real documents, ask questions, get grounded answers.

```
PDFs → PyPDFLoader → RecursiveCharacterTextSplitter → FAISS → RetrievalQA → Answers
```


In [ ]:
# !pip install pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os, glob

# ── STEP 1: Find and load all 5 patient PDFs ──────────────
# Try multiple possible locations
possible_dirs = [
    "test_patients",
    os.path.join(os.getcwd(), "test_patients"),
    os.path.join(os.path.dirname(os.getcwd()), "test_patients"),
    "/Users/ramakrishna.thirupathi/Downloads/OneDrive_1_4-3-2026/test_patients",
    "/Users/ramakrishna.thirupathi/Downloads/test_patients",
]

pdf_dir = None
for d in possible_dirs:
    if os.path.exists(d) and glob.glob(os.path.join(d, "*.pdf")):
        pdf_dir = d
        break

if pdf_dir is None:
    print("✗ test_patients/ folder not found. Searched:")
    for d in possible_dirs:
        print(f"  {d}")
    print("\nFix: copy the test_patients folder to your notebook directory.")
else:
    print(f"Found PDFs in: {pdf_dir}")

pdf_files = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf"))) if pdf_dir else []

all_docs = []
for path in pdf_files:
    loader = PyPDFLoader(path)
    docs = loader.load()
    all_docs.extend(docs)
    fname = os.path.basename(path)
    print(f"  Loaded: {fname} ({len(docs)} pages, {len(docs[0].page_content)} chars)")

print(f"\nTotal pages loaded: {len(all_docs)}")
assert len(all_docs) >= 5, f"Expected 5 PDFs, got {len(all_docs)}. Check test_patients/ folder."


**Expected Output:**
```
  Found PDFs in: test_patients
  Loaded: patient_1_pediatric.pdf (1 pages, 680 chars)
  Loaded: patient_2_pregnancy.pdf (1 pages, 720 chars)
  ...
  Total pages loaded: 5
```
If you see `✗ test_patients/ folder not found` → copy the folder next to this notebook.


In [ ]:
# ── VERIFY: Show extracted text from Patient 5 ────────────
print("=" * 60)
print("  VERIFICATION — Patient 5 Extracted Text")
print("=" * 60)

patient_5_docs = [d for d in all_docs if "patient_5" in d.metadata.get("source", "")]
if patient_5_docs:
    print(patient_5_docs[0].page_content)
    has_meds = "Clopidogrel" in patient_5_docs[0].page_content
    print(f"\n  Medications visible: {'✓ YES' if has_meds else '✗ NO — PDF extraction failed'}")
else:
    # If source metadata doesn't have filename, show last doc
    print("Showing last loaded document:")
    print(all_docs[-1].page_content)
    has_meds = "Clopidogrel" in all_docs[-1].page_content
    print(f"\n  Medications visible: {'✓ YES' if has_meds else '✗ check PDF'}")


In [ ]:
# ── STEP 2: Chunk the documents ────────────────────────────
# Patient PDFs are ~700 chars each. Use chunk_size=800 to keep each
# patient record in ONE chunk (better retrieval than splitting a short doc).
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # Keep short docs intact
    chunk_overlap=100,    # Overlap for safety
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_documents(all_docs)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunks per source:")
from collections import Counter
sources = Counter(os.path.basename(c.metadata.get('source', 'unknown')) for c in chunks)
for src, count in sources.items():
    print(f"  {src}: {count} chunks")

print(f"\nSample chunk (Patient 5):")
p5_chunks = [c for c in chunks if 'patient_5' in c.metadata.get('source', '')]
if p5_chunks:
    print(f"  {p5_chunks[0].page_content[:300]}...")


In [ ]:
# ── STEP 3-4: Embed + Index into FAISS ─────────────────────
pdf_store = FAISS.from_documents(chunks, embeddings)

print(f"PDF Vector Store created ✓")
print(f"  Chunks indexed: {pdf_store.index.ntotal}")
print(f"  Dimensions:     {pdf_store.index.d}")


In [ ]:
# ── STEP 5: Query the patient PDFs ─────────────────────────
pdf_retriever = pdf_store.as_retriever(search_kwargs={"k": 4})
pdf_rag = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=pdf_retriever,
    return_source_documents=True,
)

clinical_questions = [
    "What medications is patient 5 currently taking?",
    "Which patient has renal issues and what is their eGFR?",
    "Is there a pregnant patient? What NSAID concern exists?",
    "What drug interactions should we be concerned about for the cardiac patient?",
    "Which patient across all 5 files is at the HIGHEST risk and why?",
]

print("=" * 65)
print("  CLINICAL KNOWLEDGE ASSISTANT — Querying 5 Patient PDFs")
print("=" * 65)

for q in clinical_questions:
    result = pdf_rag.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result'][:250]}")
    sources = set()
    for doc in result['source_documents']:
        src = doc.metadata.get('source', 'unknown')
        sources.add(os.path.basename(src))
    print(f"   PDFs used: {', '.join(sources)}")


**Expected Output:**
- Q1: Lists Clarithromycin, Clopidogrel, Atorvastatin (from patient_5_cardiac.pdf)
- Q2: Patient 3 Robert Chen, eGFR 42, CKD Stage 3 (from patient_3_renal.pdf)
- Q3: Patient 2 Sarah Martinez, Naproxen concern in 3rd trimester
- Q4: CYP3A4 interaction — Clarithromycin inhibits metabolism of Clopidogrel
- Q5: Cross-document comparison — should identify Patient 4 or 5 as highest risk

**The cross-document query (Q5) is the wow moment: the agent reads ALL 5 PDFs and compares risk levels.**


---
# Part 3 — Memory Loop with Decay
### Query → Retrieve → Generate → **STORE** → **DECAY**

The agent stores each interaction. Over time, it remembers its own answers.


In [ ]:
class MemoryLoopAgent:
    def __init__(self, name, knowledge_texts):
        self.name = name
        self.store = FAISS.from_texts(knowledge_texts, embeddings)
        self.interactions = []
        self.initial_count = self.store.index.ntotal

    def respond(self, query):
        # RETRIEVE
        retrieved = self.store.similarity_search_with_score(query, k=3)
        context = "\n".join([doc.page_content for doc, _ in retrieved])

        # GENERATE
        response = llm.invoke([
            SystemMessage(content=f"You are {self.name}. Use ONLY this context:\n{context}"),
            HumanMessage(content=query)
        ]).content

        # STORE — save interaction back into vector store
        record = f"[STORED] Customer asked: {query} | Agent answered: {response[:120]}"
        self.store.add_texts([record])
        self.interactions.append({"query": query, "response": response[:120]})

        return response, retrieved

agent = MemoryLoopAgent("SupportBot", knowledge_base)

queries = [
    ("Q1", "What is the refund policy?"),
    ("Q2", "How do I track my order?"),
    ("Q3", "Can I cancel my subscription?"),
    ("Q4", "What payment methods do you accept?"),
    ("Q5", "Do you price match?"),
    ("Q6", "What did I ask you about refunds earlier?"),  # MEMORY TEST
]

print("=" * 60)
print("  MEMORY LOOP — Q6 tests recall of Q1")
print("=" * 60)

for label, q in queries:
    response, retrieved = agent.respond(q)
    print(f"\n  {label}: {q}")
    print(f"  Answer: {response[:100]}...")


In [ ]:
# ── TEST: Did Q6 retrieve the stored Q1 interaction? ──────
print("=" * 60)
print("  MEMORY RECALL TEST")
print("=" * 60)

results = agent.store.similarity_search_with_score("What did I ask about refunds?", k=5)
found_stored = False
for doc, score in results:
    is_stored = "[STORED]" in doc.page_content
    if is_stored: found_stored = True
    marker = "← STORED INTERACTION" if is_stored else "← ORIGINAL DOC"
    print(f"  [{score:.3f}] {marker}")
    print(f"           {doc.page_content[:70]}...")

print(f"\n  Memory recall: {'✓ PASS — agent found its own prior answer' if found_stored else '✗ FAIL'}")
print(f"  Store: {agent.store.index.ntotal} vectors ({agent.initial_count} docs + {len(agent.interactions)} stored)")


**Expected:** Q6 retrieves the `[STORED]` Q1 interaction. The agent found its OWN prior answer through vector similarity. That IS the memory loop.


---
# Part 3B — Model Context Protocol (MCP)
### MCP is NOT RAG — It Is the Protocol That Connects AI to ANY Tool

**MCP** (Model Context Protocol) is Anthropic's standard for how AI agents connect to **external capabilities**.

```
RAG  = one specific pattern (retrieve docs → augment prompt)
MCP  = a PROTOCOL (like HTTP) that connects AI to ANY tool
```

| MCP Component | What It Does | Example |
|--------------|-------------|---------|
| **Host** | The AI application | Your Python script, Claude Desktop, VS Code |
| **Client** | Connection manager | Sends tool requests, receives responses |
| **Server** | Exposes capabilities | A database, API, file system, calculator, vector store |
| **Tools** | Functions the AI can call | `search_patients()`, `calculate_dose()`, `check_interactions()` |
| **Resources** | Data the AI can read | Files, DB records, live feeds |

**RAG is ONE tool in the MCP toolbox. MCP is the toolbox itself.**

### Demo: An agent with 3 DIFFERENT tools (not just retrieval)


In [ ]:
# ── MCP Demo: Define 3 Different Tools ─────────────────────
# MCP lets an agent use ANY tool. Not just retrieval.
# We define 3 tools: each does something COMPLETELY different.

from langchain.tools import Tool

# TOOL 1: Patient Lookup (this is the RAG part — vector search)
def patient_lookup_fn(query: str) -> str:
    '''Search patient PDFs for relevant medical data.'''
    results = pdf_store.similarity_search_with_score(query, k=3)
    parts = []
    for doc, score in results:
        src = os.path.basename(doc.metadata.get('source', 'unknown'))
        parts.append(f"[{src} | score:{score:.3f}] {doc.page_content[:200]}")
    return "\n".join(parts)

# TOOL 2: Drug Interaction Checker (NOT retrieval — a rule-based lookup)
KNOWN_INTERACTIONS = {
    ("clarithromycin", "clopidogrel"): "SEVERE: Clarithromycin inhibits CYP3A4, reducing Clopidogrel activation. Risk: cardiac event.",
    ("clarithromycin", "atorvastatin"): "MODERATE: CYP3A4 inhibition increases Atorvastatin levels. Risk: rhabdomyolysis.",
    ("metformin", "ibuprofen"): "MODERATE: Ibuprofen may reduce renal clearance of Metformin. Risk: lactic acidosis in CKD patients.",
    ("warfarin", "aspirin"): "SEVERE: Additive anticoagulation. Risk: GI bleeding, especially with prior bleed history.",
    ("naproxen", "pregnancy"): "SEVERE: NSAIDs in 3rd trimester risk premature ductus arteriosus closure.",
}

def drug_interaction_fn(query: str) -> str:
    '''Check for known drug interactions. Input: two drug names or a drug + condition.'''
    query_lower = query.lower()
    found = []
    for (drug_a, drug_b), warning in KNOWN_INTERACTIONS.items():
        if drug_a in query_lower and drug_b in query_lower:
            found.append(f"{drug_a.title()} + {drug_b.title()}: {warning}")
    if not found:
        # Check partial matches
        for (drug_a, drug_b), warning in KNOWN_INTERACTIONS.items():
            if drug_a in query_lower or drug_b in query_lower:
                found.append(f"{drug_a.title()} + {drug_b.title()}: {warning}")
    return "\n".join(found) if found else "No known interactions found for the given query."

# TOOL 3: Dose Calculator (NOT retrieval — pure math)
def dose_calculator_fn(query: str) -> str:
    '''Calculate medication dose based on patient weight. Input: drug name and weight in kg.'''
    import re
    weight_match = re.search(r'(\d+)\s*kg', query.lower())
    weight = float(weight_match.group(1)) if weight_match else 70.0  # default 70kg

    # Standard dose-per-kg for common drugs
    dose_table = {
        "metformin": (15, "mg", "twice daily", 2000),
        "ibuprofen": (10, "mg", "every 6-8 hours", 800),
        "amoxicillin": (25, "mg", "every 8 hours", 1500),
        "methotrexate": (0.3, "mg", "once weekly", 25),
    }

    query_lower = query.lower()
    for drug, (per_kg, unit, freq, max_dose) in dose_table.items():
        if drug in query_lower:
            calc = min(per_kg * weight, max_dose)
            return (f"{drug.title()} for {weight}kg patient:\n"
                    f"  Calculated: {per_kg} {unit}/kg × {weight}kg = {per_kg * weight:.0f} {unit}\n"
                    f"  Max dose: {max_dose} {unit}\n"
                    f"  Recommended: {calc:.0f} {unit} {freq}")

    return f"Drug not found in dose table. Available: {', '.join(dose_table.keys())}"

# Register all 3 as LangChain Tools
tools = [
    Tool(name="PatientLookup",
         description="Search patient medical records from uploaded PDFs. Use when asked about patient history, symptoms, diagnoses, or current medications.",
         func=patient_lookup_fn),
    Tool(name="DrugInteractionCheck",
         description="Check for known drug-drug interactions. Use when asked if two drugs interact or if a drug combination is safe.",
         func=drug_interaction_fn),
    Tool(name="DoseCalculator",
         description="Calculate medication dose based on patient weight in kg. Use when asked about dosing or 'how much of drug X for Y kg patient'.",
         func=dose_calculator_fn),
]

print("3 MCP-style tools registered ✓")
for t in tools:
    print(f"  {t.name:25} — {t.description[:60]}...")


**3 tools, 3 completely different capabilities:**
- `PatientLookup` → vector search (this is RAG)
- `DrugInteractionCheck` → rule-based lookup (NOT retrieval)
- `DoseCalculator` → mathematical calculation (NOT retrieval)

**MCP enables ALL THREE through one protocol. RAG is just ONE of them.**


In [ ]:
# ── MCP Agent: LLM decides which tool to use ──────────────
# The agent reads the tool descriptions and picks the right one.
# This tool SELECTION is the MCP Client's core job.

print("=" * 65)
print("  MCP MULTI-TOOL AGENT — Agent Picks the Right Tool")
print("=" * 65)

test_queries = [
    ("What medications is patient 5 currently taking?",      "PatientLookup"),
    ("Does clarithromycin interact with clopidogrel?",       "DrugInteractionCheck"),
    ("Calculate metformin dose for a 94kg patient",          "DoseCalculator"),
    ("Which patient has renal problems?",                    "PatientLookup"),
    ("Is warfarin safe with aspirin?",                       "DrugInteractionCheck"),
]

for query, expected_tool in test_queries:
    print(f"\nQ: {query}")
    print(f"   Expected tool: {expected_tool}")

    # MCP Client logic: match query to tool description using LLM
    tool_descriptions = "\n".join([f"- {t.name}: {t.description}" for t in tools])
    tool_choice = llm.invoke([
        SystemMessage(content=f"You are a tool router. Given a user query, respond with ONLY the tool name (one word) that best matches.\nAvailable tools:\n{tool_descriptions}"),
        HumanMessage(content=query)
    ]).content.strip()

    print(f"   Agent picked:  {tool_choice}")

    # Find and invoke the selected tool
    selected = next((t for t in tools if t.name.lower() in tool_choice.lower()), tools[0])
    result = selected.func(query)
    print(f"   Tool output:   {result[:150]}...")

    correct = expected_tool.lower() in tool_choice.lower()
    print(f"   Correct tool:  {'✓ YES' if correct else '✗ NO'}")


In [ ]:
# ── TEST: Full MCP flow — Tool → LLM Answer ───────────────
print("=" * 65)
print("  FULL MCP FLOW: Query → Tool Selection → Tool Call → LLM Answer")
print("=" * 65)

# Complex query that needs BOTH PatientLookup AND DrugInteractionCheck
complex_q = "Patient 5 is on clarithromycin and clopidogrel. Is this combination safe?"

print(f"\nQ: {complex_q}")

# Step 1: Get patient data (Tool 1)
patient_data = tools[0].func("patient 5 medications")
print(f"\n  Tool 1 (PatientLookup): {patient_data[:150]}...")

# Step 2: Check interaction (Tool 2)
interaction = tools[1].func("clarithromycin clopidogrel")
print(f"\n  Tool 2 (DrugInteractionCheck): {interaction}")

# Step 3: LLM synthesizes both tool outputs
final_answer = llm.invoke([
    SystemMessage(content=f"You are a clinical assistant. Use these tool outputs to answer:\n\nPatient Data:\n{patient_data}\n\nDrug Interaction:\n{interaction}"),
    HumanMessage(content=complex_q)
]).content

print(f"\n  Final Answer (synthesized from 2 tools):")
print(f"  {final_answer[:350]}")

print(f"\n  ✓ The agent used 2 DIFFERENT tools and combined their outputs.")
print(f"    PatientLookup = retrieval (RAG). DrugInteractionCheck = rule lookup (NOT RAG).")
print(f"    MCP enables BOTH through one protocol. RAG is just one tool in the toolbox.")


**Expected Output:**
```
  MCP MULTI-TOOL AGENT:
  Q: What medications is patient 5 taking?     → PatientLookup     ✓
  Q: Does clarithromycin interact with ...?    → DrugInteractionCheck ✓
  Q: Calculate metformin dose for 94kg         → DoseCalculator     ✓

  FULL MCP FLOW:
  Tool 1 (PatientLookup): Clopidogrel 75mg, Aspirin 100mg, Atorvastatin...
  Tool 2 (DrugInteractionCheck): SEVERE — CYP3A4 inhibition...
  Final Answer: The combination is NOT safe. Clarithromycin inhibits CYP3A4...
```

**The key insight:**
- RAG answered "what drugs?" (retrieval from PDFs)
- Drug interaction check answered "is it safe?" (rule-based lookup — NOT retrieval)
- The agent used BOTH tools through one protocol
- **THAT is MCP: one protocol, many tools, the agent picks the right one**


---
# Part 4 — Conversational Agent with PDF Retrieval + Memory (Capstone)
### Everything combined: reads PDFs + remembers conversations

```
User turn → Retrieve from PDFs + Retrieve from memory → Generate → Store interaction → Loop
```


In [ ]:
class ClinicalAssistant:
    '''Conversational agent with PDF knowledge + conversation memory.'''

    def __init__(self, pdf_store):
        self.pdf_store = pdf_store          # Patient PDF knowledge
        self.memory_store = None            # Conversation history
        self.turn_count = 0

    def _add_memory(self, text):
        if self.memory_store is None:
            self.memory_store = FAISS.from_texts([text], embeddings)
        else:
            self.memory_store.add_texts([text])

    def chat(self, message):
        self.turn_count += 1

        # 1. RETRIEVE from PDFs (knowledge)
        pdf_results = self.pdf_store.similarity_search(message, k=3)
        pdf_context = "\n".join([d.page_content for d in pdf_results])

        # 2. RETRIEVE from conversation memory
        memory_context = "(No prior conversation)"
        if self.memory_store:
            mem_results = self.memory_store.similarity_search(message, k=3)
            if mem_results:
                memory_context = "\n".join([d.page_content for d in mem_results])

        # 3. GENERATE with both contexts
        system = f'''You are a clinical decision support assistant.

Patient Data (from uploaded PDFs):
{pdf_context}

Conversation History:
{memory_context}

Rules:
- Answer based on the patient data above
- Reference prior conversation if relevant
- If asked about drug interactions, check the medication list carefully
- Be specific: cite patient names, medication names, lab values'''

        response = llm.invoke([
            SystemMessage(content=system),
            HumanMessage(content=message)
        ]).content

        # 4. STORE this interaction
        record = f"[Turn {self.turn_count}] User: {message} | Assistant: {response[:150]}"
        self._add_memory(record)

        return response

print("ClinicalAssistant defined ✓")
print(f"PDF store: {pdf_store.index.ntotal} chunks from 5 patient PDFs")


In [ ]:
# ── 5-turn conversation about patients ─────────────────────
assistant = ClinicalAssistant(pdf_store)

turns = [
    "Tell me about patient 5 — the cardiac patient.",
    "What medications is he currently on?",
    "Are there any dangerous drug interactions I should know about?",
    "What about patient 3 — the one with kidney problems?",
    "Comparing patients 3 and 5, who is at higher immediate risk?",
]

print("=" * 65)
print("  CLINICAL ASSISTANT — Multi-turn Conversation")
print("=" * 65)

for i, msg in enumerate(turns):
    print(f"\n  You [Turn {i+1}]: {msg}")
    response = assistant.chat(msg)
    print(f"  AI: {response[:300]}")
    print(f"  {'─'*60}")


In [ ]:
# ── MEMORY TEST: Does it recall the conversation? ──────────
print("=" * 65)
print("  MEMORY TEST — Can the agent recall prior turns?")
print("=" * 65)

# Turn 6: Ask about what we discussed
memory_test = assistant.chat("What did I ask you about first in this conversation?")
print(f"\n  You: What did I ask you about first in this conversation?")
print(f"  AI:  {memory_test[:250]}")

# Turn 7: Cross-reference test
cross_test = assistant.chat("Which patient's drugs interact with each other?")
print(f"\n  You: Which patient's drugs interact with each other?")
print(f"  AI:  {cross_test[:250]}")

if assistant.memory_store:
    print(f"\n  Memory store: {assistant.memory_store.index.ntotal} interactions stored")
    print(f"  PDF store:    {assistant.pdf_store.index.ntotal} chunks")
    print(f"  Total turns:  {assistant.turn_count}")


**Expected Output:**
- Turns 1-3: Correctly describes Patient 5 (David Okafor), lists medications, identifies CYP3A4 interaction
- Turn 4: Switches to Patient 3 (Robert Chen, CKD Stage 3, eGFR 42)
- Turn 5: Compares both patients — should identify cardiac patient (drug interactions) as higher immediate risk
- Turn 6 (memory test): Recalls "you asked about patient 5, the cardiac patient"
- Turn 7 (cross-reference): Identifies Patient 5's Clarithromycin-Clopidogrel interaction

**This is a clinical decision support system: PDF retrieval + conversation memory + multi-turn reasoning.**


---
# Part 5 — Group Exercise: Build Your Domain-Specific RAG Agent
### 25 minutes — Modify the code below for YOUR domain

**Instructions:**
1. Replace `my_knowledge_base` with YOUR domain documents (10-15 chunks)
2. Build the FAISS store
3. Build the RAG + memory agent
4. Test with 3 domain-specific questions
5. Present to class (3 minutes)


In [ ]:
# ════════════════════════════════════════════════════════════
#  GROUP EXERCISE — MODIFY THIS CODE
# ════════════════════════════════════════════════════════════

# STEP 1: Replace with YOUR domain knowledge (10-15 chunks)
my_knowledge_base = [
    "YOUR DOMAIN DOC 1: Replace this with a real document from your field.",
    "YOUR DOMAIN DOC 2: Add specific policies, procedures, or facts.",
    "YOUR DOMAIN DOC 3: Include numbers, dates, names — specific details.",
    "YOUR DOMAIN DOC 4: The more specific the chunks, the better the retrieval.",
    "YOUR DOMAIN DOC 5: Each chunk should be 1-3 sentences about ONE topic.",
    # Add 5-10 more chunks...
]

# STEP 2: Build vector store
my_store = FAISS.from_texts(my_knowledge_base, embeddings)
print(f"Your store: {my_store.index.ntotal} docs indexed ✓")

# STEP 3: Build RAG + Memory Agent
class MyDomainAgent:
    def __init__(self, store):
        self.store = store
        self.memory = None

    def _add_memory(self, text):
        if self.memory is None:
            self.memory = FAISS.from_texts([text], embeddings)
        else:
            self.memory.add_texts([text])

    def ask(self, question):
        # Retrieve
        results = self.store.similarity_search(question, k=3)
        context = "\n".join([d.page_content for d in results])

        # Add memory context if exists
        mem_ctx = ""
        if self.memory:
            mem = self.memory.similarity_search(question, k=2)
            mem_ctx = "\n".join([d.page_content for d in mem])

        # Generate
        response = llm.invoke([
            SystemMessage(content=f"Answer based on:\n{context}\n\nPrior conversation:\n{mem_ctx}"),
            HumanMessage(content=question)
        ]).content

        # Store
        self._add_memory(f"Q: {question} | A: {response[:100]}")
        return response

my_agent = MyDomainAgent(my_store)

# STEP 4: Test with YOUR questions
test_questions = [
    "YOUR QUESTION 1?",
    "YOUR QUESTION 2?",
    "YOUR QUESTION 3?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {my_agent.ask(q)[:200]}...")


---
# Part 6 — End-to-End Integration Test


In [ ]:
print("=" * 60)
print("  END-TO-END INTEGRATION TEST")
print("=" * 60)

passed = failed = 0

# 1. RAG Pipeline
print("\n  [1/5] RAG Pipeline...")
try:
    r = rag_chain.invoke({"query": "refund policy"})
    assert 'result' in r and len(r['source_documents']) > 0
    passed += 1; print("        ✓ RAG answers with sources")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 2. PDF Ingestion
print("  [2/5] PDF Document Store...")
try:
    assert pdf_store.index.ntotal > 0
    r = pdf_store.similarity_search("cardiac patient", k=1)
    assert len(r) > 0
    passed += 1; print(f"        ✓ {pdf_store.index.ntotal} PDF chunks searchable")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 3. Memory Loop
print("  [3/5] Memory Loop...")
try:
    assert agent.store.index.ntotal > agent.initial_count
    passed += 1; print(f"        ✓ {len(agent.interactions)} interactions stored and retrievable")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 4. Conversational Agent
print("  [4/5] Conversational Agent...")
try:
    assert assistant.turn_count >= 5
    assert assistant.memory_store is not None
    assert assistant.memory_store.index.ntotal >= 5
    passed += 1; print(f"        ✓ {assistant.turn_count} turns with memory")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 5. PDF RAG Chain
print("  [5/5] PDF RAG Chain...")
try:
    r = pdf_rag.invoke({"query": "What drugs does the cardiac patient take?"})
    assert 'result' in r
    passed += 1; print("        ✓ PDF RAG answers clinical queries")
except Exception as e: failed += 1; print(f"        ✗ {e}")

print(f"\n  RESULTS: {passed}/{passed+failed} passed")
print(f"  {'✓ ALL TESTS PASSED — Full pipeline verified' if failed == 0 else f'✗ {failed} FAILED'}")
print("=" * 60)


**Expected:**
```
  [1/5] RAG Pipeline...           ✓
  [2/5] PDF Document Store...     ✓
  [3/5] Memory Loop...            ✓
  [4/5] Conversational Agent...   ✓
  [5/5] PDF RAG Chain...          ✓

  RESULTS: 5/5 passed
  ✓ ALL TESTS PASSED
```

---

## What You Built Today

| Component | What It Does | Real-World Equivalent |
|-----------|-------------|----------------------|
| RAG Pipeline | Grounded answers with citations | Perplexity AI ($3B) |
| PDF Ingestion | Upload docs → query instantly | Enterprise document search |
| Memory Loop | Agent remembers and improves | Netflix recommendations |
| Conversational Agent | PDF + memory in one system | Clinical decision support |

**The 5 Laws of Production Agent Memory:**
1. **PERSISTENCE** — survives restarts
2. **RETRIEVAL** — searchable by meaning
3. **DECAY** — old memories fade
4. **SEPARATION** — shared knowledge ≠ private history
5. **GOVERNANCE** — who reads, writes, deletes? Audit trail required.
